### Imports

In [1]:
from fundus_dataset import AugmentPair, FundusVesselDataset, CenterCropPair
from torch.utils.data import DataLoader
import torch.nn as nn
import torch
import matplotlib.pyplot as plt
from torch.utils.data import Subset
from monai.networks.nets import UNet
from monai.losses import DiceLoss, SoftclDiceLoss
from safetensors.torch import save_file
from pathlib import Path

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


Using device: cuda


### Testing the Dataset

In [2]:
img_dir = "fundus/train/Original/"
mask_dir = "fundus/train/Ground truth"

train_full = FundusVesselDataset(
    img_dir=img_dir,
    mask_dir=mask_dir,
    transform=AugmentPair(crop_size=(512, 512)),
)

val_full = FundusVesselDataset(
    img_dir=img_dir,
    mask_dir=mask_dir,
    transform=CenterCropPair(crop_size=(512, 512)),
)

num_samples = len(train_full)
generator = torch.Generator().manual_seed(42)

# Shuffle the indices
indices = torch.randperm(num_samples, generator=generator).tolist()

train_size = int(0.8 * num_samples)

train_indices = indices[:train_size]
val_indices = indices[train_size:]

train_dataset = Subset(train_full, train_indices)
val_dataset = Subset(val_full, val_indices)

# Dataloaders
train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    num_workers=8,
    pin_memory=torch.cuda.is_available(),
)

val_loader = DataLoader(
    val_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=8,
    pin_memory=torch.cuda.is_available(),
)

print(f"Train size: {len(train_dataset)}")
print(f"Val size: {len(val_dataset)}")

images, masks = next(iter(train_loader))
val_images, val_masks = next(iter(val_loader))

print("Train images:", images.shape, images.dtype, images.min().item(), images.max().item())
print("Train masks: ", masks.shape, masks.dtype, torch.unique(masks))

print("Val images:", val_images.shape, val_images.dtype, val_images.min().item(), val_images.max().item())
print("Val masks: ", val_masks.shape, val_masks.dtype, torch.unique(val_masks))

Number of images: 600
Number of masks: 600
First 5 image files:  ['100_A.png', '101_A.png', '102_A.png', '103_A.png', '104_A.png']
First 5 mask files:  ['100_A.png', '101_A.png', '102_A.png', '103_A.png', '104_A.png']
Number of images: 600
Number of masks: 600
First 5 image files:  ['100_A.png', '101_A.png', '102_A.png', '103_A.png', '104_A.png']
First 5 mask files:  ['100_A.png', '101_A.png', '102_A.png', '103_A.png', '104_A.png']
Train size: 480
Val size: 120


Train images: torch.Size([8, 3, 512, 512]) torch.float32 0.0 1.0
Train masks:  torch.Size([8, 1, 512, 512]) torch.float32 tensor([0., 1.])
Val images: torch.Size([1, 3, 512, 512]) torch.float32 0.0 1.0
Val masks:  torch.Size([1, 1, 512, 512]) torch.float32 tensor([0., 1.])


### CLDice + BCE + Dice

In [ ]:
import numpy as np

class DiceBCEclDiceLogitsLoss(nn.Module):
    """
    Combined Dice + BCE + Soft clDice loss
    """

    def __init__(self, iter_=3, w_dice=1.0, w_bce=1.0, w_cldice=1.0, smooth=1.0):
        super().__init__()
        self.dice = DiceLoss(sigmoid=True, smooth_nr=smooth, smooth_dr=smooth)
        self.bce = nn.BCEWithLogitsLoss()
        self.cldice = SoftclDiceLoss(iter_=iter_, smooth=smooth)
        self.w_dice = w_dice
        self.w_bce = w_bce
        self.w_cldice = w_cldice

    def forward(self, logits, masks):
        dice_loss = self.dice(logits, masks)
        bce_loss = self.bce(logits, masks)

        pred = torch.sigmoid(logits)
        pred_2ch = torch.cat([1.0 - pred, pred], dim=1)
        true_2ch = torch.cat([1.0 - masks, masks], dim=1)
        cldice_loss = self.cldice(true_2ch, pred_2ch)

        return self.w_dice * dice_loss + self.w_bce * bce_loss + self.w_cldice * cldice_loss

def hard_dice_score(logits, masks, threshold=0.5, eps=1e-6):
    probs = torch.sigmoid(logits)
    preds = (probs > threshold).float()

    # flatten each image separately: [B, 1, H, W] -> [B, pixels]
    preds = preds.flatten(start_dim=1)
    masks = masks.flatten(start_dim=1)

    intersection = (preds * masks).sum(dim=1)
    denominator = preds.sum(dim=1) + masks.sum(dim=1)

    dice = (2 * intersection + eps) / (denominator + eps)

    return dice.mean()

def hard_cldice_score(logits, masks, threshold=0.5, eps=1e-6):
    del eps  # we use this so we have same params as the hard dice
    from cldice import clDice

    probs = torch.sigmoid(logits)
    preds = (probs > threshold)
    masks = (masks > 0.5)

    # [B, 1, H, W] -> [B, H, W]
    preds = preds.squeeze(1)
    masks = masks.squeeze(1)

    cldice_scores = []
    for pred_i, mask_i in zip(preds, masks):
        pred_np = pred_i.detach().cpu().numpy().astype(np.bool_)
        mask_np = mask_i.detach().cpu().numpy().astype(np.bool_)

        if not pred_np.any() and not mask_np.any():
            score = 1.0
        else:
            score = float(clDice(pred_np, mask_np))
            if not np.isfinite(score):
                score = 0.0

        cldice_scores.append(score)

    if len(cldice_scores) == 0:
        return torch.tensor(0.0, device=logits.device)

    return torch.tensor(sum(cldice_scores) / len(cldice_scores), device=logits.device)


### U-Net Factory

In [4]:
def make_unet(channels=(16, 32, 64, 128, 256)):
    model = UNet(
        spatial_dims=2,
        in_channels=3,
        out_channels=1,
        channels=channels,
        strides=(2, 2, 2, 2),
        num_res_units=2,
    )
    return model.to(device)

### Training Loop Function

In [6]:
def validate_full_image(model, loader, loss_fn, threshold=0.5, device=torch.device("cuda")):
    model.eval()
    val_loss = 0.0
    val_dice = 0.0
    val_cldice = 0.0

    with torch.no_grad():
        for images, masks in loader:
            images = images.to(device)
            masks = masks.to(device)

            logits = model(images)
            loss = loss_fn(logits, masks)
            dice = hard_dice_score(logits, masks, threshold=threshold)
            cldice = hard_cldice_score(logits, masks, threshold=threshold)

            val_loss += loss.item() * images.size(0)
            val_dice += dice.item() * images.size(0)
            val_cldice += cldice.item() * images.size(0)

    val_loss /= len(loader.dataset)
    val_dice /= len(loader.dataset)
    val_cldice /= len(loader.dataset)
    return val_loss, val_dice, val_cldice


def train_model(model, train_loader, val_loader, loss_fn, optimizer, epochs, save_path=None, save_name="clDice.safetensors", device=torch.device("cuda")):
    history = []
    best_val_cldice = -float('inf')
    best_epoch = 0
    
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        train_loss_steps = []

        for images, masks in train_loader:
            images = images.to(device)
            masks = masks.to(device)

            optimizer.zero_grad()

            logits = model(images)
            loss = loss_fn(logits, masks)

            loss.backward()
            optimizer.step()

            step_loss = loss.item()
            train_loss += step_loss * images.size(0)
            train_loss_steps.append(step_loss)

        train_loss /= len(train_loader.dataset)

        # Default validation protocol: full-image metrics on val_loader.
        val_loss, val_dice, val_cldice = validate_full_image(
            model=model,
            loader=val_loader,
            loss_fn=loss_fn,
            threshold=0.5,
        )

        history.append({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "train_loss_steps": train_loss_steps,
            "val_loss": val_loss,
            "val_dice": val_dice,
            "val_cldice": val_cldice,
        })

        if val_cldice > best_val_cldice:
            best_val_cldice = val_cldice
            best_epoch = epoch + 1

            if save_path is not None:
                save_dir = Path(save_path)
                save_dir.mkdir(parents=True, exist_ok=True)
                st_path = save_dir / save_name
                save_file(model.state_dict(), st_path)

        print(
            f"Epoch {epoch + 1:03d}/{epochs} | "
            f"Train loss: {train_loss:.4f} | "
            f"Val loss: {val_loss:.4f} | "
            f"Val Dice: {val_dice:.4f} | "
            f"Val clDice: {val_cldice:.4f} | "
            f"Best: {best_val_cldice:.4f} @ {best_epoch}"
        )

    return history

### DANGEROUS (RESET EXPERIMENT)

In [7]:
model = make_unet(channels=(32, 64, 128, 256, 512))
loss_fn = DiceBCEclDiceLogitsLoss(w_bce=0, w_dice=0.8, w_cldice=0.2).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
EPOCHS = 100
SAVE_PATH = "/workspace/models_cldice_no_bce/"
SAVE_NAME = "clDice_no_bce.safetensors"

### START OR CONTINUE EXPERIMENT

In [8]:
history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    loss_fn=loss_fn,
    optimizer=optimizer,
    epochs=EPOCHS,
    save_path=SAVE_PATH,
    save_name=SAVE_NAME,
)

/root/Computer-Vision-2026/cldice.py:19: RuntimeWarning: invalid value encountered in scalar divide
  return np.sum(v*s)/np.sum(s)


Epoch 001/100 | Train loss: 0.7221 | Val loss: 0.7791 | Val Dice: 0.3195 | Val clDice: 0.3745 | Best: 0.3745 @ 1
Epoch 002/100 | Train loss: 0.6456 | Val loss: 0.7329 | Val Dice: 0.3108 | Val clDice: 0.5206 | Best: 0.5206 @ 2
Epoch 003/100 | Train loss: 0.6106 | Val loss: 0.7154 | Val Dice: 0.3113 | Val clDice: 0.5539 | Best: 0.5539 @ 3
Epoch 004/100 | Train loss: 0.5791 | Val loss: 0.7114 | Val Dice: 0.3237 | Val clDice: 0.5268 | Best: 0.5539 @ 3
Epoch 005/100 | Train loss: 0.5563 | Val loss: 0.7062 | Val Dice: 0.3655 | Val clDice: 0.5902 | Best: 0.5902 @ 5
Epoch 006/100 | Train loss: 0.5261 | Val loss: 0.6687 | Val Dice: 0.3643 | Val clDice: 0.5806 | Best: 0.5902 @ 5
Epoch 007/100 | Train loss: 0.4953 | Val loss: 0.6394 | Val Dice: 0.3824 | Val clDice: 0.5997 | Best: 0.5997 @ 7
Epoch 008/100 | Train loss: 0.4753 | Val loss: 0.6486 | Val Dice: 0.4122 | Val clDice: 0.5907 | Best: 0.5997 @ 7
Epoch 009/100 | Train loss: 0.4251 | Val loss: 0.6336 | Val Dice: 0.4344 | Val clDice: 0.6069 | 

### Save your Settings

In [10]:
import pandas as pd

history_df = pd.DataFrame(history)
history_df.to_csv("/workspace/models_cldice_no_bce/cldice_no_bce_history.csv", index=False)